In [ ]:
%%sql

-- ============================================================
-- GOLD FIRE TRAFFIC IMPACT
-- ============================================================
--
-- Objetivo:
--   Identificar incidencias de tráfico DGT potencialmente
--   relacionadas espacial y temporalmente con detecciones
--   de incendios NASA.
--
-- Granularidad:
--   1 detección NASA × 1 incidencia DGT
--
-- Relación espacial:
--   distancia <= 25 km
--
-- Relación temporal:
--   -3h / +3h respecto a la detección NASA
--   y/o incidencia activa en el momento de la detección.
--
-- Clave lógica:
--   fire_detection_id + traffic_incident_id
--
-- El resultado es un INDICADOR DE POSIBLE IMPACTO.
-- No afirma causalidad entre incendio e incidencia.
--
-- ============================================================


-- ------------------------------------------------------------
-- 1. Crear tabla Gold
-- ------------------------------------------------------------

CREATE TABLE IF NOT EXISTS gold_fire_traffic_impact (

    fire_detection_id STRING,

    fire_detection_timestamp TIMESTAMP,

    fire_latitude DOUBLE,
    fire_longitude DOUBLE,

    fire_brightness_ti4 DOUBLE,
    fire_brightness_ti5 DOUBLE,
    fire_radiative_power DOUBLE,
    fire_confidence STRING,
    fire_daynight STRING,

    traffic_incident_id STRING,

    traffic_road_name STRING,
    traffic_province STRING,
    traffic_autonomous_community STRING,
    traffic_municipality STRING,

    traffic_latitude DOUBLE,
    traffic_longitude DOUBLE,

    traffic_kilometer_point DOUBLE,

    traffic_cause_type STRING,
    traffic_incident_detail_type STRING,
    traffic_severity_level STRING,

    traffic_start_timestamp TIMESTAMP,
    traffic_end_timestamp TIMESTAMP,

    traffic_carriageway STRING,
    traffic_lane_usage STRING,
    traffic_vehicle_type STRING,

    distance_km DOUBLE,
    temporal_difference_minutes DOUBLE,

    incident_active_at_fire_detection BOOLEAN,

    potential_fire_impact STRING,

    impact_level STRING,

    fire_source_file STRING,
    traffic_source_file STRING,

    fire_ingestion_timestamp TIMESTAMP,
    traffic_ingestion_timestamp TIMESTAMP,

    gold_updated_at TIMESTAMP
);


-- ------------------------------------------------------------
-- 2. Detecciones NASA
-- ------------------------------------------------------------
--
-- Utilizamos el máximo detalle disponible.
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW fire_detections AS

SELECT

    SHA2(
        CONCAT_WS(
            '|',
            CAST(latitude AS STRING),
            CAST(longitude AS STRING),
            CAST(fire_detection_timestamp AS STRING)
        ),
        256
    ) AS fire_detection_id,

    fire_detection_timestamp,

    CAST(latitude AS DOUBLE) AS fire_latitude,
    CAST(longitude AS DOUBLE) AS fire_longitude,

    CAST(bright_ti4 AS DOUBLE)
        AS fire_brightness_ti4,

    CAST(bright_ti5 AS DOUBLE)
        AS fire_brightness_ti5,

    CAST(fire_radiative_power AS DOUBLE)
        AS fire_radiative_power,

    confidence AS fire_confidence,

    daynight AS fire_daynight,

    landing_source_file AS fire_source_file,

    ingestion_timestamp AS fire_ingestion_timestamp

FROM silver_nasa_fires

WHERE fire_detection_timestamp IS NOT NULL
  AND latitude IS NOT NULL
  AND longitude IS NOT NULL;


-- ------------------------------------------------------------
-- 3. Incidencias DGT
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW traffic_incidents AS

SELECT

    record_id AS traffic_incident_id,

    road_name AS traffic_road_name,

    province AS traffic_province,

    autonomous_community
        AS traffic_autonomous_community,

    municipality AS traffic_municipality,

    CAST(latitude AS DOUBLE)
        AS traffic_latitude,

    CAST(longitude AS DOUBLE)
        AS traffic_longitude,

    CAST(kilometer_point AS DOUBLE)
        AS traffic_kilometer_point,

    cause_type AS traffic_cause_type,

    incident_detail_type
        AS traffic_incident_detail_type,

    severity_level
        AS traffic_severity_level,

    start_timestamp
        AS traffic_start_timestamp,

    end_timestamp
        AS traffic_end_timestamp,

    carriageway AS traffic_carriageway,

    lane_usage AS traffic_lane_usage,

    vehicle_type AS traffic_vehicle_type,

    landing_source_file AS traffic_source_file,

    ingestion_timestamp AS traffic_ingestion_timestamp

FROM silver_dgt_traffic

WHERE record_id IS NOT NULL

  AND latitude IS NOT NULL

  AND longitude IS NOT NULL;


-- ------------------------------------------------------------
-- 4. Cruce espacial + temporal
-- ------------------------------------------------------------
--
-- Primero utilizamos un bounding box para reducir candidatos.
-- Después calculamos la distancia Haversine exacta.
--
-- Temporalmente buscamos incidencias:
--
--   fire_timestamp - 3h
--   fire_timestamp + 3h
--
-- Además identificamos si la incidencia estaba activa
-- exactamente en el momento de la detección.
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW traffic_candidates AS

SELECT

    f.fire_detection_id,

    f.fire_detection_timestamp,

    f.fire_latitude,
    f.fire_longitude,

    f.fire_brightness_ti4,
    f.fire_brightness_ti5,
    f.fire_radiative_power,
    f.fire_confidence,
    f.fire_daynight,

    f.fire_source_file,
    f.fire_ingestion_timestamp,

    t.traffic_incident_id,

    t.traffic_road_name,
    t.traffic_province,
    t.traffic_autonomous_community,
    t.traffic_municipality,

    t.traffic_latitude,
    t.traffic_longitude,

    t.traffic_kilometer_point,

    t.traffic_cause_type,
    t.traffic_incident_detail_type,
    t.traffic_severity_level,

    t.traffic_start_timestamp,
    t.traffic_end_timestamp,

    t.traffic_carriageway,
    t.traffic_lane_usage,
    t.traffic_vehicle_type,

    t.traffic_source_file,
    t.traffic_ingestion_timestamp,

    -- --------------------------------------------------------
    -- Distancia Haversine
    -- --------------------------------------------------------

    (
        6371.0 * 2.0 * ASIN(
            SQRT(

                POWER(
                    SIN(
                        RADIANS(
                            t.traffic_latitude
                            - f.fire_latitude
                        ) / 2.0
                    ),
                    2
                )

                +

                COS(
                    RADIANS(f.fire_latitude)
                )

                *

                COS(
                    RADIANS(t.traffic_latitude)
                )

                *

                POWER(
                    SIN(
                        RADIANS(
                            t.traffic_longitude
                            - f.fire_longitude
                        ) / 2.0
                    ),
                    2
                )

            )
        )
    ) AS distance_km,

    -- --------------------------------------------------------
    -- Diferencia temporal absoluta en minutos
    -- --------------------------------------------------------

    ABS(
        (
            UNIX_TIMESTAMP(
                t.traffic_start_timestamp
            )
            -
            UNIX_TIMESTAMP(
                f.fire_detection_timestamp
            )
        ) / 60.0
    ) AS temporal_difference_minutes,

    -- --------------------------------------------------------
    -- ¿La incidencia estaba activa cuando NASA detectó
    -- el incendio?
    --
    -- Si end_timestamp es NULL consideramos que continúa
    -- activa.
    -- --------------------------------------------------------

    CASE

        WHEN
            t.traffic_start_timestamp
                <= f.fire_detection_timestamp

            AND

            (
                t.traffic_end_timestamp IS NULL

                OR

                t.traffic_end_timestamp
                    >= f.fire_detection_timestamp
            )

        THEN TRUE

        ELSE FALSE

    END AS incident_active_at_fire_detection

FROM fire_detections f

INNER JOIN traffic_incidents t

    ON

        -- ----------------------------------------------------
        -- Bounding box aproximado
        -- ----------------------------------------------------

        t.traffic_latitude BETWEEN
            f.fire_latitude - 0.25
            AND
            f.fire_latitude + 0.25

        AND

        t.traffic_longitude BETWEEN
            f.fire_longitude - 0.35
            AND
            f.fire_longitude + 0.35

        -- ----------------------------------------------------
        -- Ventana temporal ±3 horas
        -- ----------------------------------------------------

        AND

        t.traffic_start_timestamp BETWEEN
            f.fire_detection_timestamp
                - INTERVAL 3 HOURS

            AND

            f.fire_detection_timestamp
                + INTERVAL 3 HOURS;


-- ------------------------------------------------------------
-- 5. Aplicar distancia real <=25 km
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW spatial_temporal_matches AS

SELECT *

FROM traffic_candidates

WHERE distance_km <= 25.0;


-- ------------------------------------------------------------
-- 6. Clasificación del posible impacto
-- ------------------------------------------------------------
--
-- Se combinan:
--
--   distancia
--   actividad temporal
--   severidad DGT
--
-- Esto es un indicador analítico, NO una prueba de causalidad.
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW classified_impacts AS

SELECT

    *,

    CASE

        WHEN incident_active_at_fire_detection
             AND distance_km <= 5
            THEN 'HIGH'

        WHEN incident_active_at_fire_detection
             AND distance_km <= 15
            THEN 'MEDIUM'

        WHEN incident_active_at_fire_detection
             AND distance_km <= 25
            THEN 'LOW'

        WHEN distance_km <= 5
            THEN 'MEDIUM'

        WHEN distance_km <= 15
            THEN 'LOW'

        ELSE 'LOW'

    END AS potential_fire_impact,

    CASE

        WHEN
            incident_active_at_fire_detection
            AND distance_km <= 5
            AND UPPER(
                COALESCE(
                    traffic_severity_level,
                    ''
                )
            ) IN (
                'HIGH',
                'SEVERE',
                'CRITICAL'
            )
            THEN 'CRITICAL'

        WHEN
            incident_active_at_fire_detection
            AND distance_km <= 5
            THEN 'HIGH'

        WHEN
            incident_active_at_fire_detection
            AND distance_km <= 15
            THEN 'MEDIUM'

        WHEN
            distance_km <= 25
            THEN 'LOW'

        ELSE 'LOW'

    END AS impact_level

FROM spatial_temporal_matches;


-- ------------------------------------------------------------
-- 7. Preparar fuente final
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW gold_fire_traffic_source AS

SELECT

    fire_detection_id,

    fire_detection_timestamp,

    fire_latitude,
    fire_longitude,

    fire_brightness_ti4,
    fire_brightness_ti5,
    fire_radiative_power,
    fire_confidence,
    fire_daynight,

    traffic_incident_id,

    traffic_road_name,
    traffic_province,
    traffic_autonomous_community,
    traffic_municipality,

    traffic_latitude,
    traffic_longitude,

    traffic_kilometer_point,

    traffic_cause_type,
    traffic_incident_detail_type,
    traffic_severity_level,

    traffic_start_timestamp,
    traffic_end_timestamp,

    traffic_carriageway,
    traffic_lane_usage,
    traffic_vehicle_type,

    distance_km,
    temporal_difference_minutes,

    incident_active_at_fire_detection,

    potential_fire_impact,

    impact_level,

    fire_source_file,
    traffic_source_file,

    fire_ingestion_timestamp,
    traffic_ingestion_timestamp,

    current_timestamp() AS gold_updated_at

FROM classified_impacts;


-- ------------------------------------------------------------
-- 8. Eliminar posibles duplicados de la fuente
-- ------------------------------------------------------------
--
-- La combinación:
--
--   fire_detection_id + traffic_incident_id
--
-- debe ser única.
--
-- Si existen varias versiones de la incidencia, conservamos
-- la más recientemente actualizada/ingerida.
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW gold_fire_traffic_deduplicated AS

SELECT *

FROM (

    SELECT

        s.*,

        ROW_NUMBER() OVER (

            PARTITION BY
                fire_detection_id,
                traffic_incident_id

            ORDER BY

                GREATEST(
                    COALESCE(
                        fire_ingestion_timestamp,
                        TIMESTAMP('1900-01-01')
                    ),
                    COALESCE(
                        traffic_ingestion_timestamp,
                        TIMESTAMP('1900-01-01')
                    )
                ) DESC

        ) AS rn

    FROM gold_fire_traffic_source s

)

WHERE rn = 1;


-- ------------------------------------------------------------
-- 9. MERGE incremental
-- ------------------------------------------------------------
--
-- Ejecutable cada 30 minutos.
--
-- MATCHED:
--     actualiza la relación existente.
--
-- NOT MATCHED:
--     crea una nueva relación incendio-incidencia.
-- ------------------------------------------------------------

MERGE INTO gold_fire_traffic_impact AS target

USING gold_fire_traffic_deduplicated AS source

ON
       target.fire_detection_id =
           source.fire_detection_id

   AND target.traffic_incident_id =
       source.traffic_incident_id


WHEN MATCHED THEN

    UPDATE SET

        target.fire_detection_timestamp =
            source.fire_detection_timestamp,

        target.fire_latitude =
            source.fire_latitude,

        target.fire_longitude =
            source.fire_longitude,

        target.fire_brightness_ti4 =
            source.fire_brightness_ti4,

        target.fire_brightness_ti5 =
            source.fire_brightness_ti5,

        target.fire_radiative_power =
            source.fire_radiative_power,

        target.fire_confidence =
            source.fire_confidence,

        target.fire_daynight =
            source.fire_daynight,

        target.traffic_road_name =
            source.traffic_road_name,

        target.traffic_province =
            source.traffic_province,

        target.traffic_autonomous_community =
            source.traffic_autonomous_community,

        target.traffic_municipality =
            source.traffic_municipality,

        target.traffic_latitude =
            source.traffic_latitude,

        target.traffic_longitude =
            source.traffic_longitude,

        target.traffic_kilometer_point =
            source.traffic_kilometer_point,

        target.traffic_cause_type =
            source.traffic_cause_type,

        target.traffic_incident_detail_type =
            source.traffic_incident_detail_type,

        target.traffic_severity_level =
            source.traffic_severity_level,

        target.traffic_start_timestamp =
            source.traffic_start_timestamp,

        target.traffic_end_timestamp =
            source.traffic_end_timestamp,

        target.traffic_carriageway =
            source.traffic_carriageway,

        target.traffic_lane_usage =
            source.traffic_lane_usage,

        target.traffic_vehicle_type =
            source.traffic_vehicle_type,

        target.distance_km =
            source.distance_km,

        target.temporal_difference_minutes =
            source.temporal_difference_minutes,

        target.incident_active_at_fire_detection =
            source.incident_active_at_fire_detection,

        target.potential_fire_impact =
            source.potential_fire_impact,

        target.impact_level =
            source.impact_level,

        target.fire_source_file =
            source.fire_source_file,

        target.traffic_source_file =
            source.traffic_source_file,

        target.fire_ingestion_timestamp =
            source.fire_ingestion_timestamp,

        target.traffic_ingestion_timestamp =
            source.traffic_ingestion_timestamp,

        target.gold_updated_at =
            source.gold_updated_at


WHEN NOT MATCHED THEN

    INSERT (

        fire_detection_id,

        fire_detection_timestamp,

        fire_latitude,
        fire_longitude,

        fire_brightness_ti4,
        fire_brightness_ti5,
        fire_radiative_power,
        fire_confidence,
        fire_daynight,

        traffic_incident_id,

        traffic_road_name,
        traffic_province,
        traffic_autonomous_community,
        traffic_municipality,

        traffic_latitude,
        traffic_longitude,

        traffic_kilometer_point,

        traffic_cause_type,
        traffic_incident_detail_type,
        traffic_severity_level,

        traffic_start_timestamp,
        traffic_end_timestamp,

        traffic_carriageway,
        traffic_lane_usage,
        traffic_vehicle_type,

        distance_km,
        temporal_difference_minutes,

        incident_active_at_fire_detection,

        potential_fire_impact,
        impact_level,

        fire_source_file,
        traffic_source_file,

        fire_ingestion_timestamp,
        traffic_ingestion_timestamp,

        gold_updated_at

    )

    VALUES (

        source.fire_detection_id,

        source.fire_detection_timestamp,

        source.fire_latitude,
        source.fire_longitude,

        source.fire_brightness_ti4,
        source.fire_brightness_ti5,
        source.fire_radiative_power,
        source.fire_confidence,
        source.fire_daynight,

        source.traffic_incident_id,

        source.traffic_road_name,
        source.traffic_province,
        source.traffic_autonomous_community,
        source.traffic_municipality,

        source.traffic_latitude,
        source.traffic_longitude,

        source.traffic_kilometer_point,

        source.traffic_cause_type,
        source.traffic_incident_detail_type,
        source.traffic_severity_level,

        source.traffic_start_timestamp,
        source.traffic_end_timestamp,

        source.traffic_carriageway,
        source.traffic_lane_usage,
        source.traffic_vehicle_type,

        source.distance_km,
        source.temporal_difference_minutes,

        source.incident_active_at_fire_detection,

        source.potential_fire_impact,
        source.impact_level,

        source.fire_source_file,
        source.traffic_source_file,

        source.fire_ingestion_timestamp,
        source.traffic_ingestion_timestamp,

        source.gold_updated_at

    );